# Paper Disruption (CD) + F/E/G + ni/nj/nk — Numba engine

For every publication, per citation window $W\in\{3,5,10,\text{all}\}$:
- **Disruption index** $CD=\dfrac{n_i-n_j}{n_i+n_j+n_k}$ — $n_i$ citers of $P$ citing none of $P$'s
  references, $n_j$ citers of $P$ also citing $\ge 1$ of them, $n_k$ papers citing $\ge 1$ reference
  of $P$ but not $P$;
- **F/E/G** (Foundation / Extension / Generalization) per citer, reported as fractions;
- **ni / nj / nk** counts.

## Input
```
Dimensions/cache/paper_graph.npz   # built by paper_citation
Dimensions/cache/paper_csr.npz     # int32 CSR (out/in adjacency) -- built here on first run
```

## Engine
The `numba` kernel of `OpenAlex/notebook/paper_disruption.ipynb`, copied verbatim (validated there
against a per-focal oracle). Only the id codec differs.

## Output
`Dimensions/output/paper_disruption.parquet` — `paper_id` + `CD/F/E/G/ni/nj/nk` × `{_3,_5,_10,_all}`.
Publications with no citers are NaN (CD/F/E/G) / −1 (ni/nj/nk).

In [1]:
import os, sys, gc, glob, time
import numpy as np, pandas as pd
import pyarrow as pa, pyarrow.parquet as pq
sys.path.insert(0, '/project/jevans/Dawoon/Science of Science/Dimensions')
import dim_common as dim
ROOT = dim.BASE; OUT = dim.OUT
print('dump:', dim.ROOT)
OUT_FP = f'{OUT}/paper_disruption.parquet'
WINS = np.array([3, 5, 10, 2_000_000_000], dtype=np.int64)
SFX  = ['_3', '_5', '_10', '_all']
# Everything below is fed by notebook/references_w_year.ipynb: it writes the per-publication
# map (year + source), the scalar and author parts, and the edge table with both years and both
# source ids. Build it once before running this notebook.
assert dim.have_consolidated(), (
    'run notebook/references_w_year.ipynb first -- it builds the map, the scalar parts and the edge table')
dim.summary()

dump: /project/jevans/dimensions/dimensions/dimensions_june_2025
dump     : /project/jevans/dimensions/dimensions/dimensions_june_2025
cache    : /project/jevans/Dawoon/Science of Science/Dimensions/cache
output   : /project/jevans/Dawoon/Science of Science/Dimensions/output
  consolidated edge table: present
  map            2.80 GB  /project/jevans/Dawoon/Science of Science/Dimensions/cache/pub_year_source_map.npz
  graph         19.00 GB  /project/jevans/Dawoon/Science of Science/Dimensions/cache/paper_graph.npz
  csr          not built  /project/jevans/Dawoon/Science of Science/Dimensions/cache/paper_csr.npz
  journal        1.24 GB  /project/jevans/Dawoon/Science of Science/Dimensions/cache/paper_journal.parquet
  fos            0.77 GB  /project/jevans/Dawoon/Science of Science/Dimensions/cache/paper_fos.parquet
  pat2pub      not built  /project/jevans/Dawoon/Science of Science/Dimensions/cache/patent2pub_edges.parquet
  scalars       4219 parts  /project/jevans/Dawoon/Science o

## 1. Load int32 CSR (build from the citation graph + cache on first run)

In [2]:
%%time
out_ptr, out_idx, in_ptr, in_idx, year, uni_mag = dim.load_csr()   # builds on first run
n = len(year)
print(f'CSR: {n:,} publications, {len(out_idx):,} edges')

graph cache present: /project/jevans/Dawoon/Science of Science/Dimensions/cache/paper_graph.npz
[915s] CSR: 155,441,856 publications, 2,141,693,663 edges -> /project/jevans/Dawoon/Science of Science/Dimensions/cache/paper_csr.npz
CSR: 155,441,856 publications, 2,141,693,663 edges


## 2. Numba engine (verbatim from the OpenAlex notebook)

In [3]:
from numba import njit, prange

@njit(inline='always')
def _bfind(arr, x):
    lo = 0; hi = len(arr)
    while lo < hi:
        mid = (lo + hi) >> 1
        if arr[mid] < x: lo = mid + 1
        else: hi = mid
    return lo < len(arr) and arr[lo] == x

@njit(parallel=True)
def compute_numba(focal, out_ptr, out_idx, in_ptr, in_idx, year, wins,
                  CD, Ff, Ef, Gf, NI, NJ, NK):
    W = len(wins)
    for t in prange(len(focal)):
        F = focal[t]
        a0 = in_ptr[F]; a1 = in_ptr[F + 1]
        if a1 == a0:
            continue
        yF = year[F]
        A = in_idx[a0:a1]
        R = out_idx[out_ptr[F]:out_ptr[F + 1]]
        Rs = np.sort(R); As = np.sort(A)
        Nw = np.zeros(W, np.int64); njw = np.zeros(W, np.int64)
        cext = np.zeros(W, np.int64); cfnd = np.zeros(W, np.int64)
        tiew = np.zeros(W, np.int64); cgw = np.zeros(W, np.int64); Bw = np.zeros(W, np.int64)
        for ci in range(len(A)):
            c = A[ci]; dc = year[c] - yF
            if dc < 0:
                continue
            up = 0; downc = np.zeros(W, np.int64)
            for ri in range(out_ptr[c], out_ptr[c + 1]):
                d = out_idx[ri]
                if _bfind(Rs, d): up += 1
                if _bfind(As, d):
                    dd = year[d] - yF
                    if dd >= 0:
                        for k in range(W):
                            if dd <= wins[k]: downc[k] += 1
            for k in range(W):
                if dc <= wins[k]:
                    Nw[k] += 1; dk = downc[k]
                    if up > 0: njw[k] += 1
                    if up > dk: cext[k] += 1
                    elif dk > up: cfnd[k] += 1
                    elif up == dk and up > 0: tiew[k] += 1
                    elif up == 0 and dk == 0: cgw[k] += 1
        totB = 0
        for ri in range(len(R)):
            r = R[ri]; totB += in_ptr[r + 1] - in_ptr[r]
        if totB > 0:
            buf = np.empty(totB, np.int32); p = 0
            for ri in range(len(R)):
                r = R[ri]
                for j in range(in_ptr[r], in_ptr[r + 1]):
                    buf[p] = in_idx[j]; p += 1
            buf.sort(); prev = np.int32(-1)
            for ii in range(totB):
                b = buf[ii]
                if b == prev or b == F: continue
                prev = b; dd = year[b] - yF
                if dd >= 0:
                    for k in range(W):
                        if dd <= wins[k]: Bw[k] += 1
        for k in range(W):
            Nk = Nw[k]; Bk = Bw[k]
            if Nk == 0 and Bk == 0: continue
            njk = njw[k]; nik = Nk - njk; nkk = Bk - njk; denom = nik + njk + nkk
            NI[k, F] = nik; NJ[k, F] = njk; NK[k, F] = nkk
            CD[k, F] = (nik - njk) / denom if denom > 0 else np.nan
            if Nk > 0:
                Ef[k, F] = (cext[k] + 0.5 * tiew[k]) / Nk
                Ff[k, F] = (cfnd[k] + 0.5 * tiew[k]) / Nk
                Gf[k, F] = cgw[k] / Nk
print('engine ready')

engine ready


## 3. Run over all focal publications with citers, then save

In [4]:
%%time
W = len(WINS)
CD = np.full((W, n), np.nan, np.float32); Ff = np.full((W, n), np.nan, np.float32)
Ef = np.full((W, n), np.nan, np.float32); Gf = np.full((W, n), np.nan, np.float32)
NI = np.full((W, n), -1, np.int32); NJ = np.full((W, n), -1, np.int32); NK = np.full((W, n), -1, np.int32)
focal_all = np.flatnonzero(in_ptr[1:] - in_ptr[:-1] > 0).astype(np.int64)
print(f'focal with citers: {len(focal_all):,}  -- warm-up compile...')
compute_numba(focal_all[:5000], out_ptr, out_idx, in_ptr, in_idx, year, WINS, CD, Ff, Ef, Gf, NI, NJ, NK)
CD[:] = np.nan; Ff[:] = np.nan; Ef[:] = np.nan; Gf[:] = np.nan; NI[:] = -1; NJ[:] = -1; NK[:] = -1
tc = time.time()
print('running full numba compute...')
compute_numba(focal_all, out_ptr, out_idx, in_ptr, in_idx, year, WINS, CD, Ff, Ef, Gf, NI, NJ, NK)
print(f'compute done in {time.time()-tc:.0f}s')

focal with citers: 83,982,116  -- warm-up compile...
running full numba compute...
compute done in 22722s


In [5]:
cols = {'paper_id': dim.code_to_id(uni_mag)}
for k, s in enumerate(SFX):
    cols[f'CD{s}'] = CD[k]; cols[f'F{s}'] = Ff[k]; cols[f'E{s}'] = Ef[k]; cols[f'G{s}'] = Gf[k]
    cols[f'ni{s}'] = NI[k]; cols[f'nj{s}'] = NJ[k]; cols[f'nk{s}'] = NK[k]
out = pd.DataFrame(cols)
out.to_parquet(OUT_FP, index=False)
print(f'WROTE {OUT_FP}  ({len(out):,} rows, {len(out.columns)} cols)')
for k, s in enumerate(SFX):
    print(f'  CD{s}: defined {np.isfinite(CD[k]).mean()*100:5.1f}%  mean {np.nanmean(CD[k]):+.4f}  |  '
          f'f/e/g = {np.nanmean(Ff[k]):.3f}/{np.nanmean(Ef[k]):.3f}/{np.nanmean(Gf[k]):.3f}')
display(out.head(10))

WROTE /project/jevans/Dawoon/Science of Science/Dimensions/output/paper_disruption.parquet  (155,441,856 rows, 29 cols)
  CD_3: defined  47.8%  mean +0.1724  |  f/e/g = 0.054/0.505/0.441
  CD_5: defined  49.7%  mean +0.2035  |  f/e/g = 0.081/0.465/0.454
  CD_10: defined  51.6%  mean +0.2341  |  f/e/g = 0.121/0.416/0.463
  CD_all: defined  54.0%  mean +0.2695  |  f/e/g = 0.158/0.368/0.474


,paper_id,CD_3,F_3,E_3,G_3,ni_3,nj_3,nk_3,CD_5,F_5,E_5,G_5,ni_5,nj_5,nk_5,CD_10,F_10,E_10,G_10,ni_10,nj_10,nk_10,CD_all,F_all,E_all,G_all,ni_all,nj_all,nk_all
0,pub.1000000001,NaN,NaN,NaN,NaN,-1,-1,-1,NaN,NaN,NaN,NaN,-1,-1,-1,NaN,NaN,NaN,NaN,-1,-1,-1,NaN,NaN,NaN,NaN,-1,-1,-1
1,pub.1000000002,-0.046729,0.178571,0.750000,0.071429,2,12,200,-0.042208,0.333333,0.571429,0.095238,4,17,287,-0.037267,0.485294,0.426471,0.088235,8,26,449,-0.022676,0.576923,0.307692,0.115385,16,36,830
2,pub.1000000003,NaN,NaN,NaN,NaN,-1,-1,-1,NaN,NaN,NaN,NaN,-1,-1,-1,NaN,NaN,NaN,NaN,-1,-1,-1,NaN,NaN,NaN,NaN,-1,-1,-1
3,pub.1000000004,1.000000,0.000000,0.000000,1.000000,1,0,0,1.000000,0.000000,0.000000,1.000000,2,0,0,1.000000,0.000000,0.000000,1.000000,4,0,0,1.000000,0.000000,0.000000,1.000000,4,0,0
4,pub.1000000005,NaN,NaN,NaN,NaN,-1,-1,-1,NaN,NaN,NaN,NaN,-1,-1,-1,NaN,NaN,NaN,NaN,-1,-1,-1,NaN,NaN,NaN,NaN,-1,-1,-1
5,pub.1000000006,-0.039301,0.105263,0.631579,0.263158,5,14,210,-0.034853,0.203704,0.611111,0.185185,7,20,346,-0.025449,0.366667,0.477778,0.155556,14,31,623,-0.007483,0.467213,0.385246,0.147541,21,40,2478
6,pub.1000000007,-0.003264,0.092308,0.892308,0.015385,1,64,19236,-0.003104,0.225000,0.765000,0.010000,1,99,31473,-0.002258,0.448980,0.520408,0.030612,22,174,67118,-0.002243,0.460199,0.509950,0.029851,22,179,69806
7,pub.1000000008,0.000000,0.000000,0.500000,0.500000,1,1,247,0.000000,0.000000,0.500000,0.500000,1,1,353,-0.008013,0.142857,0.714286,0.142857,1,6,617,-0.001903,0.136364,0.681818,0.181818,2,9,3667
8,pub.1000000009,0.000000,0.250000,0.500000,0.250000,2,2,512,-0.009950,0.142857,0.714286,0.142857,3,11,790,-0.008992,0.250000,0.634615,0.115385,6,20,1531,-0.004921,0.561644,0.383562,0.054795,23,50,5414
9,pub.1000000010,-0.005367,0.000000,0.875000,0.125000,1,7,1110,-0.005008,0.045455,0.863636,0.090909,1,10,1786,-0.004968,0.083333,0.833333,0.083333,3,21,3599,-0.003556,0.185714,0.671429,0.142857,7,28,5871
